# Quaternions and spin — the same four numbers

**The punchline.** A qubit rotation **is** a unit quaternion. Not "is analogous to", not
"is a cousin of" — the same four numbers, the same multiplication table, the same strange
half-angles. And the $-1$ that a quaternion carries silently after a $360°$ turn is
something a qubit can actually *measure*.

Quaternions were invented in 1843 for 3D geometry, eighty years before anyone had heard of
spin. Half-angles were their most-complained-about feature: to turn a vector by $\theta$ you
build a quaternion out of $\theta/2$. Textbooks call this a "quirk of the parametrisation".
It is not a quirk. It is the same fact that makes an electron need two full turns to come
back to itself, and this notebook is going to identify the two exactly, numerically, on
qsim's own gates.

Background you may want first:

- **[01 — States and gates](../01-states-and-gates.ipynb)** introduces the qubit, Dirac
  notation and the Bloch sphere from scratch.
- **[one_qubit_playground](one_qubit_playground.ipynb)** closes on the observation that
  $R_x(2\pi) = -I$ — a full turn multiplies the state by $-1$. That cell is where this
  notebook starts, and Part 5 here is the promised answer to "so is that sign *real*?"

Everything else is built from scratch, in plain NumPy, in this notebook. The route:

1. quaternions from nothing — the multiplication table, and rotating 3-vectors by sandwiching;
2. the dictionary between quaternions and one-qubit gates, checked on qsim's own $R_x$,
   $R_y$, $R_z$ — components *and* products;
3. the same rotation run twice in parallel, once as a quantum state and once as three real
   numbers pushed around by quaternion algebra;
4. the double cover: sweep to $4\pi$ and watch the two descriptions come apart;
5. the finale, where the $-1$ stops being bookkeeping and becomes a measurement outcome.

## Part 1 — Quaternions from nothing, in plain NumPy

A **quaternion** is four real numbers written as

$$q = a + b\,\mathbf{i} + c\,\mathbf{j} + d\,\mathbf{k},$$

exactly the way a complex number is two real numbers written as $a + b\,\mathbf{i}$.
Addition is componentwise and boring. Multiplication is the whole story, and it is fixed by
one line Hamilton famously carved into a Dublin bridge in 1843:

$$\mathbf{i}^2 = \mathbf{j}^2 = \mathbf{k}^2 = \mathbf{i}\mathbf{j}\mathbf{k} = -1.$$

Everything else follows from expanding that. In particular $\mathbf{ij} = \mathbf{k}$,
$\mathbf{jk} = \mathbf{i}$, $\mathbf{ki} = \mathbf{j}$ — cyclic, like a right-handed
coordinate frame — while going *backwards* costs a minus sign: $\mathbf{ji} = -\mathbf{k}$.
So quaternion multiplication is **not commutative**. That is not a defect; it is the point.
Rotations in 3D do not commute either (turn a book about $x$ then $y$, then try the other
order), so anything that represents rotations faithfully had better not commute.

Two more definitions we will need:

- the **conjugate** $\bar q = a - b\mathbf{i} - c\mathbf{j} - d\mathbf{k}$, which flips the
  three "vector" components and leaves the "scalar" component $a$ alone;
- the **norm** $\lVert q\rVert = \sqrt{a^2+b^2+c^2+d^2}$, the ordinary length of the
  4-vector. A quaternion with norm 1 is a **unit quaternion**, and those are the ones that
  represent rotations.

We store a quaternion as a plain length-4 NumPy array `[a, b, c, d]`. No classes, no
operator overloading — the arithmetic should stay visible.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsim import Circuit
from qsim.gates import H, Rx, Ry, Rz, X


def hamilton(p: np.ndarray, q: np.ndarray) -> np.ndarray:
    """The quaternion product p*q, for quaternions stored as [a, b, c, d]."""
    a1, b1, c1, d1 = p
    a2, b2, c2, d2 = q
    # Expand (a1 + b1 i + c1 j + d1 k)(a2 + b2 i + c2 j + d2 k) term by term and collect,
    # using nothing but i^2 = j^2 = k^2 = ijk = -1 and its consequences:
    #     i*j =  k,   j*k =  i,   k*i =  j       (cyclic: the right-handed order)
    #     j*i = -k,   k*j = -i,   i*k = -j       (backwards costs a minus sign)
    # The scalar row collects the three squares, which is where its minus signs come from.
    # Each vector row collects one "forwards" pair (+) and one "backwards" pair (-), which
    # is exactly the pattern of a cross product — see the markdown below.
    return np.array([
        a1 * a2 - b1 * b2 - c1 * c2 - d1 * d2,
        a1 * b2 + b1 * a2 + c1 * d2 - d1 * c2,
        a1 * c2 - b1 * d2 + c1 * a2 + d1 * b2,
        a1 * d2 + b1 * c2 - c1 * b2 + d1 * a2,
    ])


def conjugate(q: np.ndarray) -> np.ndarray:
    """The conjugate qbar: negate the i, j, k parts, keep the scalar part."""
    # Elementwise multiply by [1, -1, -1, -1]. For a *unit* quaternion the conjugate is
    # also the inverse, since q * qbar = ||q||^2 = 1.
    return q * np.array([1.0, -1.0, -1.0, -1.0])


def quat_norm(q: np.ndarray) -> float:
    """The length of the 4-vector, sqrt(a^2 + b^2 + c^2 + d^2)."""
    return float(np.sqrt(q @ q))  # q @ q is the dot product of q with itself

Before trusting `hamilton`, make it recite the multiplication table. We build the four basis
quaternions $1, \mathbf{i}, \mathbf{j}, \mathbf{k}$ as the four standard basis vectors and
multiply them together.

In [ ]:
one = np.array([1.0, 0.0, 0.0, 0.0])
qi = np.array([0.0, 1.0, 0.0, 0.0])
qj = np.array([0.0, 0.0, 1.0, 0.0])
qk = np.array([0.0, 0.0, 0.0, 1.0])

print("i*i      =", hamilton(qi, qi), "  (should be -1)")
print("j*j      =", hamilton(qj, qj), "  (should be -1)")
print("k*k      =", hamilton(qk, qk), "  (should be -1)")
print("i*j*k    =", hamilton(hamilton(qi, qj), qk), "  (should be -1)")
print()
print("i*j      =", hamilton(qi, qj), "  ( = +k)")
print("j*i      =", hamilton(qj, qi), "  ( = -k, the other way round)")
print()
print("1 is the identity:", np.allclose(hamilton(one, qj), qj))

The table checks out, and the non-commutativity is right there in the two middle lines:
$\mathbf{ij}$ and $\mathbf{ji}$ differ by a sign.

### Why unit quaternions rotate vectors — the rule, stated

Here is the trick that made quaternions famous, stated without proof (the proof is a page of
algebra and would teach you less than the numerical check that follows).

Take a 3-vector $\vec v = (v_x, v_y, v_z)$ and embed it as a **pure quaternion** — scalar
part zero, vector part $\vec v$:

$$v = 0 + v_x\mathbf{i} + v_y\mathbf{j} + v_z\mathbf{k}.$$

Take a unit axis $\hat n$ and an angle $\theta$, and build

$$q \;=\; \cos\tfrac{\theta}{2} \;+\; \sin\tfrac{\theta}{2}\,
   \bigl(n_x\mathbf{i} + n_y\mathbf{j} + n_z\mathbf{k}\bigr).$$

Then the **sandwich product**

$$v' \;=\; q\,v\,\bar q$$

comes out pure again (its scalar part is exactly zero), and its vector part is $\vec v$
rotated about $\hat n$ by $\theta$, right-hand rule.

Two things to notice before we test it.

**The half-angle is already here.** No quantum mechanics has been mentioned. This is 1843
geometry, and $\theta/2$ is already in the formula. The reason is visible in the sandwich:
$q$ appears twice, so whatever angle it carries gets applied twice — build it from
$\theta/2$ and the sandwich delivers $\theta$.

**Therefore $q$ and $-q$ do the same thing.** Negating $q$ negates $\bar q$ too, and the two
minus signs cancel inside the sandwich. So *two* unit quaternions describe every rotation.
Hold onto that; it is the whole of section 5.

The ground truth we will check against is the ordinary axis–angle rotation matrix, built by
hand from Rodrigues' formula

$$R = \cos\theta\, \mathbb{1} \;+\; \sin\theta\, K \;+\; (1 - \cos\theta)\, \hat n \hat n^{\mathsf T},$$

where $K$ is the matrix that does "cross product with $\hat n$".

In [ ]:
def axis_angle_quaternion(axis: np.ndarray, theta: float) -> np.ndarray:
    """The unit quaternion for a rotation by theta about `axis` (need not be normalised)."""
    unit = axis / np.linalg.norm(axis)
    # Scalar part cos(theta/2); vector part sin(theta/2) times the axis. The * unpacking
    # splices the three components of the array into the four-element literal.
    return np.array([np.cos(theta / 2.0), *(np.sin(theta / 2.0) * unit)])


def rotate_by_quaternion(q: np.ndarray, v: np.ndarray) -> np.ndarray:
    """Rotate the 3-vector v by the unit quaternion q, via the sandwich q v qbar."""
    pure = np.array([0.0, v[0], v[1], v[2]])          # embed v as a pure quaternion
    out = hamilton(hamilton(q, pure), conjugate(q))
    return out[1:]                                     # scalar part comes back 0; drop it


def axis_angle_matrix(axis: np.ndarray, theta: float) -> np.ndarray:
    """The 3x3 rotation matrix for the same rotation, from Rodrigues' formula."""
    n = axis / np.linalg.norm(axis)
    # K is the cross-product matrix of n: K @ v equals np.cross(n, v) for every v.
    k = np.array([[0.0, -n[2], n[1]], [n[2], 0.0, -n[0]], [-n[1], n[0], 0.0]])
    # np.outer(n, n) is the 3x3 matrix n n^T, whose action on v is n * (n . v) --
    # the projection of v onto the axis, which a rotation about that axis leaves alone.
    return (
        np.cos(theta) * np.eye(3)
        + np.sin(theta) * k
        + (1.0 - np.cos(theta)) * np.outer(n, n)
    )


rot_rng = np.random.default_rng(1843)
worst = 0.0
print(f"{'axis':>22} {'theta':>7}   {'q v qbar':>26}   {'R v':>26}")
for _ in range(4):
    axis = rot_rng.normal(size=3)
    theta = rot_rng.uniform(0.0, 2.0 * np.pi)
    v = rot_rng.normal(size=3)
    by_quaternion = rotate_by_quaternion(axis_angle_quaternion(axis, theta), v)
    by_matrix = axis_angle_matrix(axis, theta) @ v
    worst = max(worst, float(np.abs(by_quaternion - by_matrix).max()))
    unit_axis = axis / np.linalg.norm(axis)
    print(f"{np.array2string(unit_axis, precision=2):>22} {theta:7.3f}   "
          f"{np.array2string(by_quaternion, precision=3):>26}   "
          f"{np.array2string(by_matrix, precision=3):>26}")

sandwich_error = worst
print(f"\nlargest disagreement over the four trials: {sandwich_error:.2e}")

Agreement to machine precision. The sandwich really is a rotation, and building it out of
$\theta/2$ really does produce a turn by $\theta$.

That is the entire quaternion crash course. Four numbers, one multiplication rule, one
sandwich. **Nothing above knows what a qubit is.**

## Part 2 — The dictionary

Now the claim. A one-qubit gate is a $2\times2$ complex matrix — eight real numbers.
Requiring it to be **unitary** (length-preserving, so that total probability stays 1)
imposes four real conditions and leaves four. Requiring its determinant to be exactly $1$ —
a harmless normalisation that picks one representative out of each family of matrices
differing by an overall phase — costs one more, leaving three.

So one-qubit gates form a three-parameter family, with a composition law that does not
commute. And the most natural way to coordinatise it turns out to be four real numbers
subject to $a^2 + b^2 + c^2 + d^2 = 1$. We have met that object already: it was Part 1.

The dictionary is:

$$\boxed{\;U \;=\; a\,\mathbb{1} \;-\; i\bigl(b\,\sigma_x + c\,\sigma_y + d\,\sigma_z\bigr)
\qquad\longleftrightarrow\qquad q \;=\; a + b\mathbf{i} + c\mathbf{j} + d\mathbf{k}\;}$$

where $\sigma_x, \sigma_y, \sigma_z$ are the **Pauli matrices** — the three $2\times2$
matrices that measure the three Bloch-sphere axes, and whose expectation values *are* the
Bloch vector $(x, y, z)$ that `inspect.bloch_vector` returns:

$$\sigma_x = \begin{pmatrix}0&1\\1&0\end{pmatrix},\quad
\sigma_y = \begin{pmatrix}0&-i\\i&0\end{pmatrix},\quad
\sigma_z = \begin{pmatrix}1&0\\0&-1\end{pmatrix}.$$

Note the $-i$ out front. That single factor is what turns three matrices that *square to
$+1$* into three objects that *square to $-1$* — which is the defining property of
$\mathbf{i}, \mathbf{j}, \mathbf{k}$. Check it: $(-i\sigma_x)^2 = -\sigma_x^2 = -\mathbb{1}$.
And $(-i\sigma_x)(-i\sigma_y) = -\sigma_x\sigma_y = -i\sigma_z$, which is the $+\mathbf{k}$
of $\mathbf{ij} = \mathbf{k}$. The Pauli matrices are Hamilton's units in disguise, and the
disguise is one factor of $-i$.

Multiplying the dictionary out gives

$$U = \begin{pmatrix} a - i d & -c - i b \\ c - i b & a + i d\end{pmatrix},$$

so going the other way needs no linear algebra at all: the four real numbers are sitting in
the top row.

### Getting the matrices out of qsim

One wrinkle. qsim deliberately does not expose its gate matrices — the library's central
rule is that it never builds a matrix bigger than one gate, and a public "give me your
matrix" accessor would invite exactly the habit it is avoiding. So we recover them the
honest way, the way an experimentalist would: **feed the gate each basis state and record
what comes out.** $U\lvert 0\rangle$ *is* the first column of $U$; $U\lvert 1\rangle$ is the
second. Two circuit runs per matrix.

In [ ]:
def gate_matrix(apply_gate) -> np.ndarray:
    """The 2x2 matrix of a one-qubit gate, read out of qsim one column at a time.

    `apply_gate` is a function taking a qubit handle, e.g. `lambda q: Rx(q, theta=0.4)`.
    """
    columns = []
    for basis_bit in (0, 1):
        qc = Circuit(name="probe", seed=0)
        q = qc.alloc("q")
        if basis_bit:
            X(q)              # prepare |1> instead of |0>
        apply_gate(q)
        columns.append(qc.inspect.state_vector())
    # U|0> and U|1> are the first and second columns of U, so stacking the two output
    # state vectors side by side (axis=1 makes them columns, not rows) rebuilds U.
    return np.stack(columns, axis=1)


PAULI_X = np.array([[0, 1], [1, 0]], dtype=complex)
PAULI_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
PAULI_Z = np.array([[1, 0], [0, -1]], dtype=complex)


def from_quaternion(q: np.ndarray) -> np.ndarray:
    """The 2x2 unitary matrix for the quaternion q = a + bi + cj + dk."""
    a, b, c, d = q
    return a * np.eye(2, dtype=complex) - 1j * (b * PAULI_X + c * PAULI_Y + d * PAULI_Z)


def to_quaternion(u: np.ndarray) -> np.ndarray:
    """The quaternion [a, b, c, d] of a 2x2 unitary, read straight off its entries.

    Since  U = [[a - i d,  -c - i b],
                [c - i b,   a + i d]],
    the top row alone determines all four numbers.
    """
    return np.array([u[0, 0].real, -u[0, 1].imag, -u[0, 1].real, -u[0, 0].imag])


# Sanity check that the two directions are inverse to each other.
sample_q = axis_angle_quaternion(np.array([1.0, 2.0, -3.0]), 0.9)
print("round trip q -> U -> q reproduces q:",
      np.allclose(to_quaternion(from_quaternion(sample_q)), sample_q))

### Prediction 1 — every rotation gate is a unit quaternion, with the axis you expect

If the dictionary is right, then `Rx(theta)` — whose whole job is "rotate the Bloch vector
by $\theta$ about $x$" — must map to *precisely* the axis–angle quaternion of Part 1:

$$R_x(\theta) \;\longleftrightarrow\; \cos\tfrac{\theta}{2} + \sin\tfrac{\theta}{2}\,\mathbf{i},$$

and likewise $R_y \to \mathbf{j}$, $R_z \to \mathbf{k}$. Note that we are *not* going to
hand the quaternion any $\theta/2$: we hand it $\theta$, ask the same rotation of both, and
see whether the half-angle appears on its own.

In [ ]:
gate_axes = {
    "Rx": (Rx, np.array([1.0, 0.0, 0.0])),
    "Ry": (Ry, np.array([0.0, 1.0, 0.0])),
    "Rz": (Rz, np.array([0.0, 0.0, 1.0])),
}
angles = [0.0, np.pi / 4, np.pi / 2, np.pi, 3.0 * np.pi / 2, 2.0 * np.pi]

component_error = 0.0
norm_error = 0.0

print(f"{'gate':>5} {'theta':>7}   {'a':>8} {'b':>8} {'c':>8} {'d':>8}   {'||q||':>6}"
      f"   {'cos(t/2)':>9} {'sin(t/2)':>9}")
for name, (gate, axis) in gate_axes.items():
    for theta in angles:
        u = gate_matrix(lambda q, g=gate, t=theta: g(q, theta=t))
        q = to_quaternion(u)
        predicted = axis_angle_quaternion(axis, theta)
        component_error = max(component_error, float(np.abs(q - predicted).max()))
        norm_error = max(norm_error, abs(quat_norm(q) - 1.0))
        print(f"{name:>5} {theta:7.3f}   {q[0]:8.4f} {q[1]:8.4f} {q[2]:8.4f} {q[3]:8.4f}"
              f"   {quat_norm(q):6.4f}   {np.cos(theta / 2):9.4f} {np.sin(theta / 2):9.4f}")

print(f"\nlargest deviation from (cos(theta/2), sin(theta/2) * axis): {component_error:.2e}")
print(f"largest deviation of ||q|| from 1:                          {norm_error:.2e}")

Read the last two columns against the first four. `Rx` puts $\cos(\theta/2)$ in $a$ and
$\sin(\theta/2)$ in $b$ — the $\mathbf{i}$ slot — and nothing anywhere else. `Ry` fills
$\mathbf{j}$, `Rz` fills $\mathbf{k}$. Every one has norm exactly 1.

**This is the answer to "why does the Bloch sphere use half-angles?"** It is not a
convention someone chose, and it is not a quantum peculiarity. Rotations of 3D space are
*already* parametrised by half-angles the moment you insist on representing them by
something that multiplies associatively and never degenerates — which is what Hamilton
found in 1843 and what the Pauli matrices rediscovered in 1927. The half-angle was in the
geometry the whole time. Quantum mechanics just handed us a physical system whose state
lives in the four-number layer rather than the three-number one.

Look at the last row of each block, $\theta = 2\pi$: the quaternion is $(-1, 0, 0, 0)$. That
is $-1$, not $1$. A full turn.

### Prediction 2 — matrix product $=$ Hamilton product

A dictionary between two sets of objects is only interesting if it also translates the
*verbs*. The verb here is composition: doing one rotation after another. On the matrix side
that is $U_1 U_2$; on the quaternion side it is `hamilton(q1, q2)`. The claim is that they
are the same operation, with the same ordering convention (rightmost acts first).

If this holds, the map is a **group isomorphism**: unit quaternions and the group $SU(2)$ of
one-qubit gates are literally the same group wearing different notation.

The test: 300 random pairs of qsim rotation gates, with angles drawn from $[0, 4\pi)$ so
that the double-cover region gets exercised too.

In [ ]:
pair_rng = np.random.default_rng(2024)
gate_list = [Rx, Ry, Rz]
product_error = 0.0

for _ in range(300):
    g1 = gate_list[pair_rng.integers(3)]
    g2 = gate_list[pair_rng.integers(3)]
    t1 = float(pair_rng.uniform(0.0, 4.0 * np.pi))
    t2 = float(pair_rng.uniform(0.0, 4.0 * np.pi))
    u1 = gate_matrix(lambda q, g=g1, t=t1: g(q, theta=t))
    u2 = gate_matrix(lambda q, g=g2, t=t2: g(q, theta=t))
    # Translate-then-multiply versus multiply-then-translate. If the two agree for every
    # pair, the dictionary is a homomorphism, not just a coincidence of components.
    via_matrices = to_quaternion(u1 @ u2)
    via_quaternions = hamilton(to_quaternion(u1), to_quaternion(u2))
    product_error = max(product_error, float(np.abs(via_matrices - via_quaternions).max()))

print(f"largest disagreement over 300 random pairs: {product_error:.3e}")

# One concrete example, printed in full, so the claim is not just a max-error number.
ux = gate_matrix(lambda q: Rx(q, theta=0.7))
uy = gate_matrix(lambda q: Ry(q, theta=1.9))
print("\nRx(0.7) then Ry(1.9)")
print("  quaternion of the product matrix :",
      np.array2string(to_quaternion(uy @ ux), precision=6))
print("  Hamilton product of the two      :",
      np.array2string(hamilton(to_quaternion(uy), to_quaternion(ux)), precision=6))
print("  ...and in the other order        :",
      np.array2string(hamilton(to_quaternion(ux), to_quaternion(uy)), precision=6))

Agreement to $10^{-15}$ across 300 random pairs, and the third line shows the two orders
genuinely differ — the non-commutativity survives the translation, as it must.

So: **the group of one-qubit gates and the group of unit quaternions are the same group.**
Not similar. The same. Every fact you know about one is a fact about the other.

## Part 3 — Both of them rotate the same vector

An isomorphism of groups is an algebraic statement. Here is the geometric one, which is
what you would actually feel if you could hold a qubit.

Prepare a qubit in some arbitrary state with a seeded pile of `Ry`/`Rz` rotations, and read
its Bloch vector once with `inspect.bloch_vector`. From that moment we run two experiments
side by side:

- **qsim**, applying real rotation gates to the real quantum state and reporting the Bloch
  vector after each;
- **NumPy**, taking that one recorded 3-vector and rotating it by quaternion sandwiches —
  never looking at the quantum state again.

The NumPy track never resynchronises. If the two agree after six chained rotations, they
agree because the quaternion sandwich *is* what a rotation gate does to a Bloch vector.

In [ ]:
qc = Circuit(name="spin", seed=11)
q = qc.alloc("q")

prep_rng = np.random.default_rng(7)
for _ in range(3):                              # an arbitrary starting orientation
    Ry(q, theta=float(prep_rng.uniform(0.0, 2.0 * np.pi)))
    Rz(q, theta=float(prep_rng.uniform(0.0, 2.0 * np.pi)))

predicted_v = np.array(qc.inspect.bloch_vector(q))   # the one and only reading we borrow
print("starting Bloch vector:", np.array2string(predicted_v, precision=4))
print()

steps = []
bloch_error = 0.0
print(f"{'step':>4} {'gate':>4} {'theta':>7}   {'qsim (x, y, z)':>28}   "
      f"{'quaternion (x, y, z)':>28}")
for step in range(6):
    name = ["Rx", "Ry", "Rz"][step % 3]
    gate, axis = gate_axes[name]
    theta = float(prep_rng.uniform(0.0, 2.0 * np.pi))

    gate(q, theta=theta)                                     # the quantum track
    measured_v = np.array(qc.inspect.bloch_vector(q))

    u = gate_matrix(lambda w, g=gate, t=theta: g(w, theta=t))
    predicted_v = rotate_by_quaternion(to_quaternion(u), predicted_v)   # the numpy track

    bloch_error = max(bloch_error, float(np.abs(measured_v - predicted_v).max()))
    steps.append((name, theta, measured_v, predicted_v))
    print(f"{step:>4} {name:>4} {theta:7.3f}   "
          f"{np.array2string(measured_v, precision=4):>28}   "
          f"{np.array2string(predicted_v, precision=4):>28}")

print(f"\nlargest disagreement after six chained rotations: {bloch_error:.2e}")

The two right-hand columns are the same numbers to every digit printed. The same picture,
with the eighteen numbers side by side: bar height is what qsim's quantum state says, the
cross is what Hamilton's arithmetic predicted.

In [ ]:
# One bar per (step, component): the bar height is qsim's Bloch component, the cross is
# what the quaternion sandwich predicted. Eighteen pairs, no visible daylight between them.
positions = np.arange(18)
qsim_values = np.concatenate([measured for _, _, measured, _ in steps])
quat_values = np.concatenate([predicted for _, _, _, predicted in steps])

fig, ax = plt.subplots(figsize=(10.0, 3.6))
ax.bar(positions, qsim_values, width=0.62, color="#8fb8d8",
       edgecolor="#3f6f96", label="qsim: inspect.bloch_vector")
ax.plot(positions, quat_values, "kx", markersize=8, markeredgewidth=1.6,
        linestyle="none", label=r"numpy: $q\,v\,\bar q$")
ax.axhline(0.0, color="gray", lw=0.6)
for boundary in range(1, 6):
    ax.axvline(3 * boundary - 0.5, color="gray", lw=0.5, linestyle=":")
ax.set_xticks(positions, [c for _ in steps for c in "xyz"])
ax.set_xlabel("Bloch component, grouped by rotation step (0 … 5)")
ax.set_ylim(-1.35, 1.45)
ax.legend(fontsize=9, ncol=2, loc="upper center")
ax.set_title("Six chained rotations: the quantum state and a bare 3-vector stay in step")
fig.tight_layout()

Every cross lands on top of its bar, to $10^{-15}$ or so. The NumPy track has not seen a
quantum state since the first line; it has been pushing three real numbers around with
Hamilton's 1843 arithmetic, and it tracks the qubit exactly.

This is worth stating carefully, because it is the point where a beginner usually stops
being confused about the Bloch sphere. The Bloch sphere is *not* a picture invented to make
qubits look friendly. A qubit's state space really is the unit quaternions, and the Bloch
sphere is what you see when you look at that space through the sandwich $q v \bar q$ — the
map that forgets exactly one thing. What it forgets is the subject of the next section.

## Part 4 — The double cover: what the sandwich forgets

We noted in Part 1 that $q$ and $-q$ produce the same rotation, because the sandwich uses
$q$ twice. Run that observation as an experiment.

Sweep $\theta$ from $0$ to $4\pi$ — two full turns — for $R_x$, and plot two things on one
figure:

- the **Bloch vector** the qubit actually has, which is what any measurement could tell you;
- the **quaternion components** $a = \cos(\theta/2)$ and $b = \sin(\theta/2)$, which is what
  the state vector is doing underneath.

In [ ]:
sweep = np.linspace(0.0, 4.0 * np.pi, 241)
bloch_rows = []
quat_rows = []

for theta in sweep:
    qc_sweep = Circuit(name="cover", seed=0)
    qq = qc_sweep.alloc("q")
    Rx(qq, theta=float(theta))
    bloch_rows.append(qc_sweep.inspect.bloch_vector(qq))
    quat_rows.append(to_quaternion(gate_matrix(lambda w, t=theta: Rx(w, theta=float(t)))))

# Lists of tuples become (241, 3) and (241, 4) arrays; column k is one component
# as a function of theta.
bloch_sweep = np.array(bloch_rows)
quat_sweep = np.array(quat_rows)

fig, ax = plt.subplots(figsize=(9.5, 4.0))
ax.plot(sweep, bloch_sweep[:, 2], lw=2.4, color="#3f6f96", label="Bloch z  (period 2π)")
ax.plot(sweep, bloch_sweep[:, 1], lw=2.4, color="#8fb8d8", label="Bloch y  (period 2π)")
ax.plot(sweep, quat_sweep[:, 0], "k--", lw=1.6, label=r"quaternion $a=\cos(\theta/2)$")
ax.plot(sweep, quat_sweep[:, 1], ":", color="#c33b53", lw=1.8,
        label=r"quaternion $b=\sin(\theta/2)$")
ax.axvline(2.0 * np.pi, color="#c33b53", lw=1.0)
ax.axhline(0.0, color="gray", lw=0.6)
ax.annotate("one full turn: Bloch home, quaternion at −1",
            xy=(2.0 * np.pi, -1.02), xytext=(2.0 * np.pi + 0.35, -1.5), fontsize=9,
            ha="left", va="center", color="#c33b53",
            arrowprops={"arrowstyle": "->", "color": "#c33b53", "lw": 1.0})
ax.set_xticks([0, np.pi, 2 * np.pi, 3 * np.pi, 4 * np.pi], ["0", "π", "2π", "3π", "4π"])
ax.set_xlabel(r"$\theta$ in $R_x(\theta)$")
ax.set_ylim(-1.8, 1.65)
ax.legend(fontsize=9, ncol=2, loc="upper center")
ax.set_title(r"$SU(2) \to SO(3)$ is two-to-one: the sphere comes home twice as often")
fig.tight_layout()

print("at theta = 2π  the quaternion is", np.array2string(quat_sweep[120], precision=6))
print("at theta = 4π  the quaternion is", np.array2string(quat_sweep[240], precision=6))

The two solid curves — everything observable — complete a cycle at $2\pi$ and do it again.
The two dashed curves do not: at $\theta = 2\pi$ the quaternion sits at $(-1, 0, 0, 0)$,
which is $-1$, and only at $4\pi$ does it return to $+1$.

The name for this is the **double cover**. The map from unit quaternions (equivalently
$SU(2)$, equivalently one-qubit gates) to ordinary 3D rotations ($SO(3)$) is two-to-one:
$q$ and $-q$ send down to the same rotation. Going once around a loop of rotations lifts to
a path that ends at $-q$; going around twice brings you back.

Two ways to say the same thing:

- **Geometrically.** The space of 3D rotations is not simply connected — there is a loop in
  it that cannot be shrunk to a point, and a rotation by $2\pi$ traverses it. Quaternion
  space is the simply-connected space sitting above it, wrapped twice.
- **Physically.** $R_x(2\pi) = -\mathbb{1}$, exactly as the closing cell of
  [one_qubit_playground](one_qubit_playground.ipynb) found. **$q$ and $-q$ are the same
  rotation but they are not the same quantum operation.**

That second sentence is the crux, and it is where most treatments stop, with a note that the
$-1$ is "just a global phase, unobservable". The next section takes that seriously and then
breaks it.

## Part 5 — Cashing out the minus sign

First, the honest part. On a single qubit, all by itself, the $-1$ really is invisible.

Every probability is $|{\rm amplitude}|^2$, and $|-z|^2 = |z|^2$. Multiply the whole state
by $-1$ and every measurement statistic on every axis is untouched. Below, two circuits — one
that has been turned through $2\pi$ and one that has not — sampled from the same seed.

In [ ]:
def turned_qubit(theta: float) -> Circuit:
    """One qubit, rotated about x by theta. Same seed every time."""
    qc_one = Circuit(name="phase", seed=5)
    qubit = qc_one.alloc("q")
    Rx(qubit, theta=theta)
    return qc_one


turned = turned_qubit(2.0 * np.pi)   # a full 360-degree turn
still = turned_qubit(0.0)            # nothing at all

print("after a 2π turn :", turned.inspect.ket())
print("after nothing   :", still.inspect.ket())
print()
print("Bloch vectors identical:",
      np.allclose(turned.inspect.bloch_vector(turned.qubits[0]),
                  still.inspect.bloch_vector(still.qubits[0])))
turned_counts = turned.inspect.sample(2000)
still_counts = still.inspect.sample(2000)
print("2000 samples each, tallies literally equal:", turned_counts == still_counts)
print("  turned:", dict(turned_counts), " still:", dict(still_counts))

The state vectors are visibly different — $-1.000\lvert 0\rangle$ against
$1.000\lvert 0\rangle$ — and no experiment on this qubit can tell them apart. So far the
textbook is right.

### Now make it a *relative* phase

The trick is the one every quantum algorithm runs on, and the one
[one_qubit_playground](one_qubit_playground.ipynb) built its interferometer from: a phase
that is invisible on the whole state becomes an *outcome* the moment it applies to only part
of a superposition.

So we rotate the qubit through $2\pi$ **conditionally**. Put a control qubit $c$ into
$\lvert +\rangle$ with a Hadamard, and run the $2\pi$ turn only on the branch where
$c = 1$:

```python
H(c)
with qc.control(c):
    Rx(t, theta=2*np.pi)
H(c)
```

Follow the amplitudes. After the first Hadamard the two qubits are in

$$\tfrac{1}{\sqrt2}\bigl(\lvert 0\rangle_c + \lvert 1\rangle_c\bigr)\otimes\lvert 0\rangle_t.$$

The controlled block applies $\mathbb{1}$ to the target on the $c=0$ branch and
$R_x(2\pi) = -\mathbb{1}$ on the $c=1$ branch. The target itself is unchanged either way —
it is still $\lvert 0\rangle_t$, and it never becomes entangled with anything. But the $c=1$
branch has picked up a factor of $-1$:

$$\tfrac{1}{\sqrt2}\bigl(\lvert 0\rangle_c - \lvert 1\rangle_c\bigr)\otimes\lvert 0\rangle_t.$$

That is $\lvert -\rangle$, not $\lvert +\rangle$. The control has been flipped from one
equator point to the opposite one — by a rotation applied to a *different qubit*, which
returned that qubit to exactly where it started. The final Hadamard turns
$\lvert -\rangle$ into $\lvert 1\rangle$, and the measurement reads **1, every time**.

Do the same with $R_x(4\pi) = +\mathbb{1}$ and the control stays $\lvert +\rangle$, the final
Hadamard returns it to $\lvert 0\rangle$, and the measurement reads **0, every time**.

In [ ]:
def belt_trick(theta: float, seed: int = 17) -> tuple[Circuit, float]:
    """H-sandwich on a control, with an Rx(theta) applied to the target in between.

    Returns the circuit and the exact probability that the control reads 1.
    """
    qc_pair = Circuit(name="belt", seed=seed)
    c = qc_pair.alloc("c")
    t = qc_pair.alloc("t")
    H(c)                             # control into |+>: both branches now exist
    with qc_pair.control(c):
        Rx(t, theta=theta)           # turn the target, but only on the c = 1 branch
    H(c)                             # recombine the branches and read the phase
    # probabilities() is indexed by the basis state as an integer, qubit 0 (the control)
    # most significant. So indices 2 and 3 are |10> and |11>: the control reading 1.
    probs = qc_pair.inspect.probabilities()
    return qc_pair, float(probs[2] + probs[3])


def measure_control(theta: float, seed: int) -> int:
    """Run the experiment once from scratch and measure the control qubit for real."""
    fresh, _ = belt_trick(theta, seed=seed)
    return fresh.measure(fresh.qubits[0])   # qubits[0] is that circuit's own control


for label, theta in (("Rx(2π) = -1  (one turn) ", 2.0 * np.pi),
                     ("Rx(4π) = +1  (two turns)", 4.0 * np.pi),
                     ("no rotation at all      ", 0.0)):
    circuit, p_one = belt_trick(theta)
    # Five independent runs, five different seeds -- so a lucky random stream cannot be
    # what makes the answer come out the same each time.
    outcomes = [measure_control(theta, seed) for seed in range(5)]
    print(f"{label}  state {circuit.inspect.ket()!s:>16}   "
          f"P(control = 1) = {p_one:.12f}   five measurements: {outcomes}")

p_one_turn = belt_trick(2.0 * np.pi)[1]
p_two_turns = belt_trick(4.0 * np.pi)[1]

$P = 1$ against $P = 0$. Not a shifted distribution, not a statistical hint — two
deterministic, opposite answers, separated by nothing but whether the target qubit was
turned through $360°$ or $720°$.

Sit with what that means. The target qubit ends in exactly the state it started in. Its
Bloch vector never moved by the end; its density matrix is unchanged; every measurement you
could perform *on it* is unaffected. In the language of Part 4, the two experiments differ by
a rotation that is the **identity element of $SO(3)$**. And yet the control qubit comes out
in an orthogonal state, and a measurement tells you which one happened, every single shot.

**The $-1$ is physical.** Quaternion users carry it as bookkeeping — a sign that cancels in
the sandwich and is conventionally ignored. A qubit hands it to you as a measurement outcome.

The classical shadow of this is the **belt trick** (or plate trick, or Dirac scissors): hold
one end of a belt, rotate the other end through $360°$, and the belt is twisted; rotate
through another $360°$ in the *same* direction and, remarkably, the twist can be worked out
without turning either end again. The belt is tracking a path in rotation space, not just an
endpoint, and paths of length $2\pi$ and $4\pi$ are genuinely different. Objects that
transform this way are called **spinors**, and every electron, proton and neutron in your
body is one. Neutron interferometry measured exactly the experiment above in 1975 — the
$2\pi$-rotated beam came back out of phase with its unrotated twin, and the interference
fringes moved.

The punchline, restated: **the sign quaternions carry silently, a qubit can cash out.**

## Where to go next

- **[one_qubit_playground](one_qubit_playground.ipynb)** — the $R_x(2\pi) = -I$ observation
  this notebook set out to explain, plus the interferometer that turns any phase difference
  into a probability.
- **[01 — States and gates](../01-states-and-gates.ipynb)** — the Bloch sphere from scratch,
  if the geometry above went past too quickly.
- **[04 — Combinators](../04-combinators.ipynb)** — `with qc.control(c):` as a general tool,
  and why controlling a *block* rather than a gate is the interesting move. Phase kickback,
  which is what Part 5 is a minimal instance of, is the engine of phase estimation and
  therefore of Shor's algorithm.
- **[02 — Entanglement](../02-entanglement.ipynb)** — Part 5's controlled rotation left the
  two qubits *unentangled*, which is unusual and is exactly why the phase stayed readable.
  Entangling versions of the same move are where the Bloch vector starts to shrink.

## Assertions

Every claim above, re-checked numerically.

In [ ]:
# 1. The quaternion sandwich is the axis-angle rotation matrix (Part 1).
assert sandwich_error < 1e-12, sandwich_error

# 2. Every rotation gate maps to a UNIT quaternion whose components are exactly
#    (cos(theta/2), sin(theta/2) * axis) -- checked across three gates and six angles.
assert component_error < 1e-12, component_error
assert norm_error < 1e-12, norm_error
for rotation, unit_axis in gate_axes.values():
    for angle in [0.3, 1.0, np.pi, 2.5 * np.pi]:
        matrix = gate_matrix(lambda w, g=rotation, t=angle: g(w, theta=t))
        assert np.allclose(to_quaternion(matrix),
                           axis_angle_quaternion(unit_axis, angle), atol=1e-12)
        assert abs(quat_norm(to_quaternion(matrix)) - 1.0) < 1e-12

# 3. Matrix product <-> Hamilton product, over 300 random pairs (Part 2).
assert product_error < 1e-12, product_error

# 4. Quaternion conjugation rotates the Bloch vector exactly as the gates do (Part 3).
assert bloch_error < 1e-10, bloch_error

# 5. A full 2*pi turn is the quaternion -1: scalar part -1, vector part zero (Part 4).
full_turn = to_quaternion(gate_matrix(lambda w: Rx(w, theta=2 * np.pi)))
assert np.allclose(full_turn, [-1.0, 0.0, 0.0, 0.0], atol=1e-12), full_turn
double_turn = to_quaternion(gate_matrix(lambda w: Rx(w, theta=4 * np.pi)))
assert np.allclose(double_turn, [1.0, 0.0, 0.0, 0.0], atol=1e-12), double_turn
#    ...while the Bloch vector is 2*pi-periodic: the sweep's value at 2π equals its value at 0.
assert np.allclose(bloch_sweep[120], bloch_sweep[0], atol=1e-12)
assert np.allclose(bloch_sweep[240], bloch_sweep[0], atol=1e-12)

# 6. And that sign is an observable: the H-sandwich reads it out deterministically (Part 5).
assert abs(p_one_turn - 1.0) < 1e-12, p_one_turn
assert abs(p_two_turns - 0.0) < 1e-12, p_two_turns
assert abs(belt_trick(0.0)[1] - 0.0) < 1e-12

# 7. ...but on the qubit alone it is not: identical statistics, opposite state vectors.
assert turned_counts == still_counts
assert np.allclose(turned.inspect.state_vector(), -still.inspect.state_vector())

print("all assertions passed")